In [2]:
from pathlib import Path
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import concord as ccd
import anndata as ad
import copy
import matplotlib.pyplot as plt
import time
from scipy.stats import linregress
import scipy
import sys

In [3]:
print('pyhon version:')
sys.version

pyhon version:


'3.14.3 | packaged by conda-forge | (main, Feb  9 2026, 21:56:48) [MSC v.1944 64 bit (AMD64)]'

In [4]:
torch.__version__

'2.10.0+cpu'

In [5]:
np.__version__

'2.4.2'

In [6]:
ccd.__version__

'1.0.13'

In [7]:
pd.__version__

'2.3.3'

In [8]:
sc.__version__

C:\Users\jonat\AppData\Local\Temp\ipykernel_19976\4147423214.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  sc.__version__


'1.12'

In [9]:
scipy.__version__

'1.17.1'

In [10]:
ad.__version__


C:\Users\jonat\AppData\Local\Temp\ipykernel_19976\3903165381.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ad.__version__


'0.12.10'

In [11]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.rcParams['svg.fonttype'] = 'none'

In [12]:
download_base = Path('data/abc_atlas')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

'releases/20260415/manifest.json'

List the all of the different releases available and usable by the cache object we have just loaded.

#### Downloading individual expression matrix files

In [13]:
cortex_matrix_labels = ['WMB-10Xv2-Isocortex-1/raw']

In [14]:
sc_path = abc_cache.get_file_path(directory='WMB-10Xv2', file_name=cortex_matrix_labels[0])

WMB-10Xv2-Isocortex-1-raw.h5ad:   0%|          | 0.00/8.60G [00:00<?, ?MB/s]

WMB-10Xv2-Isocortex-1-raw.h5ad: 100%|██████████| 8.60G/8.60G [06:19<00:00, 22.7MMB/s]    


In [15]:
icortexv2_1 = ad.read_h5ad(sc_path)

In [16]:
pia_layer_cells = pd.read_csv("rs_seqFISH_brain_all_fov/PIA_layer_cells.csv")

In [17]:
pia_layer_cells['cell'] = pia_layer_cells['pos_cell'] 

In [18]:
pia_layer_cells.set_index(['pos', 'cell'], inplace=True)

In [19]:
gxcm = pd.read_csv("rs_seqFISH_brain_all_fov/cell_x_gene_matrix.csv") ##"sfmb_cbgm.csv")

In [20]:
gxcm.set_index(['pos', "cell"], inplace=True)
gxcm

Aars  Aco2  Actn4  Aebp1  Aqp1  Arf1  Arpc2  Arpc5  Cct3  Cct5  ...  \
pos cell                                                                  ...   
0   5      1.0   4.0    7.0    0.0   5.0   1.0    4.0    1.0   3.0   1.0  ...   
    8      4.0   9.0    2.0    1.0  21.0   8.0    2.0    1.0   8.0   3.0  ...   
    9      1.0   9.0    8.0    4.0  12.0   7.0    3.0    2.0   4.0   6.0  ...   
    10     1.0   2.0    0.0    0.0   7.0   0.0    0.0    0.0   1.0   6.0  ...   
    11     3.0   4.0    4.0    0.0  10.0   4.0    2.0    0.0   7.0   6.0  ...   
...        ...   ...    ...    ...   ...   ...    ...    ...   ...   ...  ...   
76  83     6.0   4.0   17.0    0.0   4.0  27.0   15.0    3.0   6.0   0.0  ...   
    84     3.0   1.0    5.0    0.0   1.0   7.0    6.0    0.0   0.0   0.0  ...   
    85     0.0   0.0    0.0    0.0   0.0   2.0    3.0    0.0   0.0   0.0  ...   
    87     0.0   0.0    1.0    0.0   0.0   0.0    1.0    1.0   0.0   0.0  ...   
    88     0.0   0.0    0.0    0.0   0.0   0.0    0.0    0.0   0.0   0.0  ...   

          Xdh  Ybx1  Ywhae  Ywhag  Ywhah  Ywhaq  Ywhaz  Zfp36l1  Zfp36l2  Zyx  
pos cell                                                                       
0   5     0.0   1.0    1.0    2.0    7.0    0.0    6.0      5.0      1.0  4.0  
    8     1.0   0.0    0.0    5.0    3.0    1.0   11.0      2.0      5.0  0.0  
    9     0.0   1.0    7.0    8.0    7.0    3.0   11.0      2.0      3.0  3.0  
    10    0.0   0.0    0.0    6.0    5.0    0.0    3.0      1.0      0.0  0.0  
    11    2.0   0.0    0.0    5.0    5.0    0.0    4.0      8.0      3.0  2.0  
...       ...   ...    ...    ...    ...    ...    ...      ...      ...  ...  
76  83    0.0   0.0    2.0   52.0    0.0    1.0   66.0      3.0      0.0  2.0  
    84    0.0   1.0    1.0   14.0    0.0    0.0   27.0      1.0      0.0  0.0  
    85    0.0   0.0    0.0    1.0    0.0    0.0    2.0      0.0      0.0  0.0  
    87    0.0   0.0    0.0    0.0    0.0    0.0    3.0      0.0      0.0  0.0  
    88    0.0   0.0    0.0    0.0    0.0    0.0    1.0      0.0      0.0  0.0  

[9138 rows x 271 columns]

In [21]:
pia_cell_indices = pd.merge(gxcm, pia_layer_cells, left_index=True, right_index=True, how='inner').index

In [22]:
gxcm.drop(pia_cell_indices, inplace=True)

In [23]:
gxcm_filt = gxcm.loc[gxcm.sum(axis=1) > 300]

In [24]:
rssf = ad.AnnData(gxcm_filt.reset_index(drop=True))

c:\Users\jonat\miniconda3\envs\concord_brain_atlas\Lib\functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [25]:
rssf.obs['pos'] = np.array(gxcm_filt.reset_index()["pos"])
rssf.obs['cell'] = np.array(gxcm_filt.reset_index()["cell"])

In [26]:
icortexv2_1_genes = copy.deepcopy(icortexv2_1.var)

In [27]:
icortexv2_1_genes_gs_ind = icortexv2_1_genes.reset_index().set_index("gene_symbol")

In [28]:
gene_ids = icortexv2_1_genes_gs_ind.loc[rssf.var.index]['gene_identifier']

In [29]:
sc.pp.filter_cells(icortexv2_1, min_genes=10)
sc.pp.normalize_total(icortexv2_1)

In [30]:
icortexv2_1_rssf_genes = icortexv2_1[:, gene_ids]

In [31]:
icortexv2_1

AnnData object with n_obs × n_vars = 250040 × 32285
    obs: 'cell_barcode', 'library_label', 'anatomical_division_label', 'n_genes'
    var: 'gene_symbol'

In [32]:
icortexv2_1 

AnnData object with n_obs × n_vars = 250040 × 32285
    obs: 'cell_barcode', 'library_label', 'anatomical_division_label', 'n_genes'
    var: 'gene_symbol'

In [36]:
icortexv2_1.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1068605082 stored elements and shape (250040, 32285)>

In [33]:
icortexv2_1.X.sum(axis=1).mean()

np.float32(11284.0)

In [34]:
icortexv2_1_rssf_genes.X.sum(axis=1).mean()

np.float32(564.3048)